In [0]:
df_quality_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_data/quality_events_raw.csv")
)

display(df_quality_bronze)

In [0]:
df_quality_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pharma_bronze.bronze_quality_events")

In [0]:
from pyspark.sql.functions import current_timestamp

df_quality_silver = (
    df_quality_bronze
    .withColumn("ingestion_timestamp", current_timestamp())
)

display(df_quality_silver)

In [0]:
from delta.tables import DeltaTable

delta_quality_target = DeltaTable.forName(
    spark,
    "workspace.pharma_silver.silver_quality_events"
)

(
    delta_quality_target.alias("target")
    .merge(
        df_quality_silver.alias("source"),
        "target.quality_event_id = source.quality_event_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
from pyspark.sql.functions import col

print("Total quality events:", df_quality_silver.count())

print(
    "Duplicate quality event IDs:",
    df_quality_silver
    .groupBy("quality_event_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Null quality event IDs:",
    df_quality_silver
    .filter(col("quality_event_id").isNull())
    .count()
)

In [0]:
df_quality_final = spark.table(
    "workspace.pharma_silver.silver_quality_events"
)

print(
    "Quality Events Silver records:",
    df_quality_final.count()
)

print(
    "Duplicate quality event IDs:",
    df_quality_final
    .groupBy("quality_event_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

In [0]:
null_quality_ids = df_quality_silver.filter(
    col("quality_event_id").isNull()
).count()

print("NULL Quality Event IDs:", null_quality_ids)

In [0]:
duplicate_quality_ids = (
    df_quality_silver
    .groupBy("quality_event_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Quality Event IDs:", duplicate_quality_ids)

In [0]:
df_quality_silver.select("event_status").distinct().show()

In [0]:
valid_event_statuses = [
    "Resolved",
    "Closed",
    "Open",
    "Investigating"
]

invalid_event_status_count = df_quality_silver.filter(
    ~col("event_status").isin(valid_event_statuses)
).count()

print("Invalid Event Statuses:", invalid_event_status_count)